# Bagging-based feature selection

This notebook finds a feature set independently for every power-quality disturbance (PQD) class. It follows the idea in `Vossi_01_Task.ipynb`: start with all eight features and remove features that do not improve classification, while evaluating the decision across bootstrap samples and multiple classifiers.

The word *bagging* refers to bootstrap aggregation of the evaluation. Each candidate feature set is trained on a sample drawn with replacement and scored on its out-of-bag (OOB) samples. Each class is evaluated as a one-vs-rest binary problem, so the output preserves class-specific feature choices.


## 1. Imports and paths

The helper module contains reusable data-loading and feature-selection functions. The path setup makes the notebook work both from `Code/01 Task` and from the repository root.


In [1]:
from pathlib import Path
import sys
import pandas as pd

TASK_DIR = Path.cwd()
if TASK_DIR.name != "01 Task":
    TASK_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "01 Task")
sys.path.insert(0, str(TASK_DIR))

from bagging_feature_selection import (
    load_dataset,
    majority_vote_features,
    run_feature_selection,
)


## 2. Load and inspect the data

Each CSV file represents one disturbance class; its filename supplies the class label. The loader standardizes the columns to `F1` through `F8`, adds the source noise level, and combines all requested SNR folders. The noise-level column is retained for diagnostics but is not passed to the classifiers.


In [2]:
folders = [TASK_DIR / name for name in (
    "SNR_20db", "SNR_30db", "SNR_40db", "SNR_50db", "SNR_noiseless"
)]
data = load_dataset(folders)
print(f"Loaded {len(data):,} samples from {data['noise_level'].nunique()} noise levels")
data.head()


Loaded 6,705 samples from 5 noise levels


,F1,F2,F3,F4,F5,F6,F7,F8,label,noise_level
0,0.006464,0.004608,1.0028,0.98885,5,0.006183,0.042333,3.8125,Flicker,SNR_20db
1,0.007683,0.008596,1.0251,0.99548,6,0.017805,0.032209,0.0000,Flicker,SNR_20db
2,0.005503,0.010356,1.0279,0.99049,6,0.008724,0.056552,0.0000,Flicker,SNR_20db
3,0.005579,0.027732,1.0454,0.95545,4,0.007886,0.046220,7.3200,Flicker,SNR_20db
4,0.006768,0.026114,1.0381,0.95490,4,0.007208,0.030164,2.2875,Flicker,SNR_20db


## 3. Select features for every class and classifier

For each class and classifier pair, the selector creates a one-vs-rest target: the selected class is positive and all other classes are negative. It starts with all eight features, evaluates every possible one-feature removal using bootstrap OOB F1, accepts the best removal when it is not meaningfully worse than the current set, and repeats until no further removal is supported or one feature remains.

The scaler is inside the estimator pipeline, so it is fitted separately for every bootstrap training sample. This prevents OOB information from leaking into preprocessing. `min_improvement=0.002` prevents tiny score fluctuations from causing unstable removals. Increase `n_bootstraps` for a final experiment.


In [3]:
results = run_feature_selection(
    data,
    n_bootstraps=20,
    min_improvement=0.002,
    random_state=42,
)

# The function returns one list containing every class/classifier result.
assert isinstance(results, list), (
    "Restart the kernel to load the updated feature-selection module."
)
print(f"Generated {len(results)} class/classifier feature selections")

summary = pd.DataFrame({
    "target_label": [result.target_label for result in results],
    "classifier": [result.classifier for result in results],
    "selected_features": [list(result.selected_features) for result in results],
    "bootstrap_f1": [result.score for result in results],
})
summary


Generated 18 class/classifier feature selections


,target_label,classifier,selected_features,bootstrap_f1
0,Flicker,logistic_regression,"[F1, F2, F3, F5, F6, F7]",0.551607
1,Flicker,linear_svm,"[F1, F2, F3, F4, F6, F7, F8]",0.566215
2,Harmonics,logistic_regression,"[F1, F2, F3, F4, F7, F8]",0.958840
3,Harmonics,linear_svm,"[F1, F2, F3, F4, F7, F8]",0.978402
4,Interruption,logistic_regression,"[F1, F3, F6, F8]",0.968052
5,Interruption,linear_svm,"[F1, F3, F6, F8]",0.967289
6,Normal,logistic_regression,"[F1, F2, F3, F5, F6, F7, F8]",0.683489
7,Normal,linear_svm,"[F1, F2, F3, F6, F7, F8]",0.695970
8,Sag,logistic_regression,"[F1, F4, F6, F8]",0.896315
9,Sag,linear_svm,"[F1, F2, F3, F4, F5, F6, F7, F8]",0.900013


## 4. Inspect the per-class recommendations

The table contains one row for every class and classifier, so selected features can differ between disturbance types. The majority vote below combines only the classifier results for the same class. A feature must be selected by a strict majority of classifiers for that class to be recommended.


In [4]:
for target_label in summary["target_label"].unique():
    print(f"\n{target_label}")
    display(summary[summary["target_label"] == target_label])

recommended_features = majority_vote_features(results)
recommendation_table = pd.DataFrame({
    "target_label": list(recommended_features),
    "recommended_features": [
        list(features) for features in recommended_features.values()
    ],
})
recommendation_table



Flicker


,target_label,classifier,selected_features,bootstrap_f1
0,Flicker,logistic_regression,"[F1, F2, F3, F5, F6, F7]",0.551607
1,Flicker,linear_svm,"[F1, F2, F3, F4, F6, F7, F8]",0.566215



Harmonics


,target_label,classifier,selected_features,bootstrap_f1
2,Harmonics,logistic_regression,"[F1, F2, F3, F4, F7, F8]",0.958840
3,Harmonics,linear_svm,"[F1, F2, F3, F4, F7, F8]",0.978402



Interruption


...

,target_label,classifier,selected_features,bootstrap_f1
12,Swell,logistic_regression,"[F1, F3, F4, F5]",0.955950
13,Swell,linear_svm,"[F1, F2, F3, F4, F5, F7, F8]",0.958414



Swell_harmonics


,target_label,classifier,selected_features,bootstrap_f1
14,Swell_harmonics,logistic_regression,"[F1, F2, F3, F4, F5, F6, F7, F8]",0.776560
15,Swell_harmonics,linear_svm,"[F1, F2, F3, F4, F5, F6, F7, F8]",0.799565



Transient


,target_label,classifier,selected_features,bootstrap_f1
16,Transient,logistic_regression,"[F1, F3, F4, F5, F6, F7, F8]",0.922819
17,Transient,linear_svm,"[F1, F2, F3, F5, F6, F7]",0.927214


## 5. Interpret the output

The summary reports the selected subset and mean bootstrap one-vs-rest F1 for each classifier and target class. `recommended_features` is the final per-class output: each entry is the feature set selected by a strict majority of classifiers for that class. A high F1 score means that the selected class can be separated from the other disturbance types using that subset.

Use `recommended_features["Sag"]` (for example) when building a class-specific downstream model. If the classifiers disagree completely, the helper returns all tied most-frequently selected features so every class still receives a usable recommendation. For a final reported result, rerun with a larger `n_bootstraps` value and keep the random seed fixed so the experiment remains reproducible.
